In [1]:
import pandas as pd
df = pd.read_parquet('/kaggle/input/notebooks/ajax0564/vyom-ai-gflinear2-final-data/multiclass_dataset.parquet')

df['is_multilabel'] = False
df.head(1)

,text,label,all_label,is_multilabel
0,... a delicious crime drama on par with the sl...,[very positive],"[very positive, positive, neutral, negative, v...",False


In [2]:
df1 = pd.read_parquet('/kaggle/input/notebooks/ajax0564/vyom-ai-gflinear2-final-data/multilabel_dataset.parquet')
df1 = df1.rename(columns={'labels':'label','all_labels':'all_label'})
df1['is_multilabel'] = True
df1.head()

,text,label,all_label,is_multilabel
0,Standard cosmological models fail to account f...,"[Energy Science, Astrophysics, Materials Scien...","[Genomics, Astrophysics, Materials Science, Ne...",True
1,Rising costs in advanced manufacturing are for...,"[Robotics, Electoral Politics & Campaigns]","[Artificial Intelligence, Cloud Computing, Cyb...",True
2,"I don't do the things you mentioned above, I j...","[disapproval, gratitude]","[admiration, amusement, anger, annoyance, appr...",True
3,Configure a comprehensive investigation into t...,"[Professional League Governance, Financial Tec...","[Artificial Intelligence, Cloud Computing, Cyb...",True
4,"Under cryogenic pressures exceeding 200 GPa, d...","[Astrophysics, Climate Science, Nuclear Physics]","[Genomics, Astrophysics, Materials Science, Ne...",True


In [3]:
df_all = pd.concat([df, df1], ignore_index=True)
df_all = df_all.sample(frac = 1)
df_all.shape

(157576, 4)

In [4]:
df_all.iloc[0]['is_multilabel']

np.False_

In [5]:
set1 = df_all.sample(frac=0.9, random_state=42)

# Set 2: Sample 3 different random rows from the remaining data
remaining_df = df_all.drop(set1.index)
set2 = remaining_df.sample(frac=0.1, random_state=42)

set1.to_parquet('multilabel_dataset_train.parquet',index=False)

set2.to_parquet('multilabel_dataset_val.parquet',index=False)

In [ ]:
# %%writefile train_script.py

import math
import random
from dataclasses import dataclass, fields
from typing import Any, Dict, List, Optional, Union

import pyarrow.parquet as pq
import torch
import torch.nn as nn
from accelerate import Accelerator, DataLoaderConfiguration, PartialState, DistributedDataParallelKwargs
from accelerate.utils import set_seed
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F
import math
# Configure how data loaders are handled
dataloader_config = DataLoaderConfiguration(dispatch_batches=False)

random.seed(42)
torch.manual_seed(42)


@dataclass(frozen=True)
class SpecialTokens:
    SEP_STRUCT: str = "[SEP_STRUCT]"
    SEP_TEXT: str = "[SEP_TEXT]"
    P_TOKEN: str = "[P]"
    C_TOKEN: str = "[C]"
    E_TOKEN: str = "[E]"
    R_TOKEN: str = "[R]"
    L_TOKEN: str = "[L]"
    EXAMPLE_TOKEN: str = "[EXAMPLE]"
    OUTPUT_TOKEN: str = "[OUTPUT]"
    DESC_TOKEN: str = "[DESCRIPTION]"

    @property
    def SPECIAL_TOKENS(self) -> List[str]:
        """Returns all string field values as a list."""
        return [getattr(self, f.name) for f in fields(self)]


def sample_labels(
    all_candidate_labels: List[str],
    correct_labels: List[str],
    max_negatives: int = 10,
) -> List[str]:
    """Subsamples negative labels while retaining ground-truth positive labels with strict deduplication."""
    correct_set = list(dict.fromkeys(correct_labels))
    negative_candidates = [
        label for label in dict.fromkeys(all_candidate_labels) if label not in set(correct_set)
    ]

    num_negatives = min(len(negative_candidates), max_negatives)
    sampled_negatives = random.sample(negative_candidates, num_negatives)

    combined_labels = correct_set + sampled_negatives
    random.shuffle(combined_labels)
    return combined_labels


def format_input(
    text: str,
    candidate_labels: List[str],
    special_tokens: Any,
    is_multilabel: bool,
) -> str:
    task_type = "topics:" if is_multilabel else "sentiment:"
    label_prefix = " ".join(
        [f"{special_tokens.L_TOKEN} {label}" for label in candidate_labels]
    )
    
    # Place task_type before candidate labels so the last label ends cleanly at SEP_TEXT
    prefix = f"{special_tokens.P_TOKEN} {task_type} {label_prefix}"
    full_text = f"{prefix} {special_tokens.SEP_TEXT} {text}"
    return full_text


class ParquetIterableClassificationDataset(IterableDataset):
    def __init__(
        self,
        parquet_path: str,
        tokenizer: Any,
        special_tokens: Any,
        max_negatives: Optional[int] = 10,
        batch_size: int = 64,
    ):
        self.parquet_path = parquet_path
        self.tokenizer = tokenizer
        self.special_tokens = special_tokens
        self.max_negatives = max_negatives
        self.batch_size = batch_size
        
        # Cache row count once at initialization
        pf = pq.ParquetFile(self.parquet_path)
        self._num_rows = pf.metadata.num_rows

    def __len__(self) -> int:
        return self._num_rows

    def __iter__(self):
        pf = pq.ParquetFile(self.parquet_path)
        num_row_groups = pf.num_row_groups

        # 1. Shard across GPUs / DDP processes
        state = PartialState()
        per_process = int(math.ceil(num_row_groups / float(state.num_processes)))
        process_start = state.process_index * per_process
        process_end = min(process_start + per_process, num_row_groups)
        process_row_groups = list(range(process_start, process_end))

        # 2. Shard across DataLoader workers (if num_workers > 0)
        worker_info = get_worker_info()
        if worker_info is None:
            row_groups = process_row_groups
        else:
            num_process_groups = len(process_row_groups)
            per_worker = int(
                math.ceil(num_process_groups / float(worker_info.num_workers))
            )
            worker_id = worker_info.id
            worker_start = worker_id * per_worker
            worker_end = min(worker_start + per_worker, num_process_groups)
            row_groups = process_row_groups[worker_start:worker_end]

        l_token_id = self.tokenizer.convert_tokens_to_ids(
            self.special_tokens.L_TOKEN
        )

        for rg_idx in row_groups:
            row_group = pf.read_row_group(rg_idx)
            for batch in row_group.to_batches(max_chunksize=self.batch_size):
                pydict = batch.to_pydict()
                texts = pydict["text"]
                labels_list = pydict["all_label"]
                correct_labels = pydict["label"]
                is_multilabel_col = pydict["is_multilabel"]

                for text, all_labels, ground_truth, is_multi in zip(
                    texts, labels_list, correct_labels, is_multilabel_col
                ):
                    ground_truth = list(dict.fromkeys(ground_truth))
                    all_labels = list(dict.fromkeys(all_labels))

                    if is_multi and self.max_negatives is not None:
                        candidate_labels = sample_labels(
                            all_candidate_labels=all_labels,
                            correct_labels=ground_truth,
                            max_negatives=self.max_negatives,
                        )
                    else:
                        candidate_labels = sample_labels(
                            all_candidate_labels=all_labels,
                            correct_labels=ground_truth,
                            max_negatives=5,
                        )
                        

                    full_text = format_input(
                        text=text,
                        candidate_labels=candidate_labels,
                        special_tokens=self.special_tokens,
                        is_multilabel=is_multi,
                    )

                    encoding = self.tokenizer(
                        full_text,
                        truncation=True,
                        return_tensors=None,
                        max_length=384,
                    )
                    input_ids = encoding["input_ids"]

                    targets = [
                        1.0 if label in ground_truth else 0.0
                        for label in candidate_labels
                    ]

                    yield {
                        "input_ids": torch.tensor(
                            input_ids, dtype=torch.long
                        ),
                        "is_multilabel": is_multi,
                        "targets": targets,
                    }


def collate_fn(batch, pad_token_id=0):
    input_ids_list = []
    attention_mask_list = []
    flat_targets = []
    is_multilabel_list = []
    num_labels_per_sample = []

    for item in batch:
        encoding = item["input_ids"]
        if isinstance(encoding, dict):
            input_ids = encoding["input_ids"]
            mask = encoding.get(
                "attention_mask", torch.ones_like(input_ids)
            )
        else:
            input_ids = encoding
            mask = torch.ones_like(input_ids)

        if not isinstance(input_ids, torch.Tensor):
            input_ids = torch.tensor(input_ids, dtype=torch.long)
        if not isinstance(mask, torch.Tensor):
            mask = torch.tensor(mask, dtype=torch.long)

        input_ids_list.append(input_ids)
        attention_mask_list.append(mask)

        flat_targets.extend(item["targets"])
        is_multilabel_list.append(item["is_multilabel"])
        num_labels_per_sample.append(len(item["targets"]))

    padded_input_ids = pad_sequence(
        input_ids_list, batch_first=True, padding_value=pad_token_id
    )
    padded_attention_mask = pad_sequence(
        attention_mask_list, batch_first=True, padding_value=0
    )

    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "targets": torch.tensor(flat_targets, dtype=torch.float32),
        "is_multilabel": torch.tensor(is_multilabel_list, dtype=torch.bool),
        "num_labels_per_sample": num_labels_per_sample,
    }

class GLiNERTextClassifier(nn.Module):
    def __init__(
        self,
        base_model: nn.Module,
        label_token_id: int,
        sep_text_token_id: int,
        projection_dim: int = 256,
    ):
        super().__init__()
        self.encoder = base_model
        self.hidden_size = self.encoder.config.hidden_size
        self.label_token_id = label_token_id
        self.sep_text_token_id = sep_text_token_id

        # Text & Label projection heads
        self.text_proj = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(self.hidden_size, projection_dim),
        )
        self.label_proj = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(self.hidden_size, projection_dim),
        )

        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1.0 / 0.07))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Extract Text Representations (tokens after [SEP_TEXT])
        
        sep_mask = (input_ids == self.sep_text_token_id)
        after_sep_mask = torch.cumsum(sep_mask.long(), dim=1) > 0
        text_mask = after_sep_mask & (~sep_mask) & (attention_mask == 1)

        has_sep = sep_mask.any(dim=1, keepdim=True)
        valid_text_mask = torch.where(has_sep, text_mask, attention_mask == 1)

        masked_text_hidden = sequence_output * valid_text_mask.unsqueeze(-1)
        text_sums = masked_text_hidden.sum(dim=1)
        token_counts = valid_text_mask.sum(dim=1, keepdim=True).clamp(min=1)
        text_reps = text_sums / token_counts  # [batch_size, hidden_size]

        
        # Vectorized Span Mean-Pooling for Label Tokens ([L] + words)
        
        label_start_mask = (input_ids == self.label_token_id)
        num_total_labels = label_start_mask.sum().item()

        if num_total_labels == 0:
            return torch.empty(0, device=input_ids.device)

        # Region for labels: tokens occurring after the first [L] and before [SEP_TEXT]
        before_sep = (torch.cumsum(sep_mask.long(), dim=1) == 0) & (~sep_mask)
        after_first_label = torch.cumsum(label_start_mask.long(), dim=1) > 0
        # [L] A B [L] C D [SEP_TEXT]
        # valid_label_mask = before_sep & after_first_label & (~label_start_mask) & (attention_mask == 1) #label = mean( A, B) and label 1 = mean( C, D)
        valid_label_mask = before_sep & after_first_label & (attention_mask == 1) #label = mean([L], A, B) and label 1 = mean([L], C, D)

        # Map each label token to a 0-indexed global label ID across the batch
        flat_starts = label_start_mask.view(-1).long()
        global_label_ids = (torch.cumsum(flat_starts, dim=0) - 1).view(input_ids.shape)

        # Extract features and IDs of label tokens
        valid_features = sequence_output[valid_label_mask]  # [num_label_tokens, hidden_size]
        valid_ids = global_label_ids[valid_label_mask]      # [num_label_tokens]

        # Scatter-add token embeddings and count occurrences per label ID
        label_sums = torch.zeros(
            num_total_labels, self.hidden_size, device=input_ids.device, dtype=sequence_output.dtype
        )
        label_sums.index_add_(0, valid_ids, valid_features)

        label_counts = torch.zeros(
            num_total_labels, 1, device=input_ids.device, dtype=sequence_output.dtype
        )
        label_counts.index_add_(
            0, valid_ids, torch.ones_like(valid_ids, dtype=sequence_output.dtype).unsqueeze(-1)
        )

        label_features = label_sums / label_counts.clamp(min=1e-8)  # [num_total_labels, hidden_size]

        
        # Align Text Representations with Label Batch Indices
        
        batch_indices = label_start_mask.nonzero(as_tuple=True)[0]
        aligned_text_reps = text_reps[batch_indices]  # [num_total_labels, hidden_size]

        
        # Projections & Scaled Similarity Logits
        
        proj_text = F.normalize(self.text_proj(aligned_text_reps), dim=-1)
        proj_labels = F.normalize(self.label_proj(label_features), dim=-1)

        scale = self.logit_scale.exp().clamp(max=100.0)
        similarity_logits = (proj_text * proj_labels).sum(dim=-1) * scale

        return similarity_logits

class FocalLossWithLogits(nn.Module):
    """
    Binary Focal Loss for logits.
    Down-weights easy negatives using a modulating factor (1 - p_t)^gamma.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Compute standard binary cross-entropy with logits per element (no reduction)
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        # Get probabilities
        probas = torch.sigmoid(logits)
        
        # p_t is p for true class, 1-p for false class
        p_t = probas * targets + (1 - probas) * (1 - targets)
        
        # Alpha balancing factor
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        
        # Final focal loss
        loss = alpha_t * focal_weight * bce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class HybridClassificationLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        # Replaced BCEWithLogitsLoss with FocalLossWithLogits
        self.focal = FocalLossWithLogits(alpha=alpha, gamma=gamma, reduction='mean')
        self.ce = nn.CrossEntropyLoss()

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        num_labels_per_sample: List[int],
        is_multilabel: torch.Tensor,
    ) -> torch.Tensor:
        offset = 0
        sample_losses = []

        for b, is_multi in enumerate(is_multilabel):
            n_labels = num_labels_per_sample[b]
            if n_labels == 0:
                continue

            sample_logits = logits[offset : offset + n_labels]
            sample_targets = targets[offset : offset + n_labels]

            if is_multi:
                # Use Focal Loss instead of BCE
                loss = self.focal(sample_logits, sample_targets)
            else:
                # Check if positive ground truth exists before calling CrossEntropy
                has_positive = (sample_targets == 1.0).any()
                if has_positive:
                    target_idx = torch.argmax(sample_targets)
                    loss = self.ce(sample_logits.unsqueeze(0), target_idx.unsqueeze(0))
                else:
                    # Fallback to Focal Loss if truncation dropped the positive candidate
                    loss = self.focal(sample_logits, sample_targets)

            sample_losses.append(loss)
            offset += n_labels

        if not sample_losses:
            return logits.sum() * 0.0

        return torch.stack(sample_losses).mean()
        
@torch.no_grad()
def evaluate(
    model: nn.Module,
    val_dataloader: DataLoader,
    accelerator: Accelerator,
    tokenizer: Any,
    special_tokens: Any,
    criterion: nn.Module,
    threshold: float = 0.5,
    num_samples_to_print: int = 10,
) -> Dict[str, float]:
    model.eval()

    total_loss = torch.tensor(0.0, device=accelerator.device)
    total_samples = torch.tensor(0.0, device=accelerator.device)

    tp = torch.tensor(0.0, device=accelerator.device)
    fp = torch.tensor(0.0, device=accelerator.device)
    fn = torch.tensor(0.0, device=accelerator.device)
    tn = torch.tensor(0.0, device=accelerator.device)

    sample_predictions = []
    special_ids_set = set(tokenizer.all_special_ids)
    label_token_id = tokenizer.convert_tokens_to_ids(special_tokens.L_TOKEN)

    for batch in val_dataloader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        targets = batch["targets"]
        num_labels_per_sample = batch["num_labels_per_sample"]
        is_multilabel_tensor = batch["is_multilabel"]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        loss = criterion(
            logits=logits,
            targets=targets,
            num_labels_per_sample=num_labels_per_sample,
            is_multilabel=is_multilabel_tensor,
        )

        batch_num_samples = len(is_multilabel_tensor)
        total_loss += loss.detach() * batch_num_samples
        total_samples += batch_num_samples

        is_multilabel_list = (
            is_multilabel_tensor.tolist()
            if isinstance(is_multilabel_tensor, torch.Tensor)
            else is_multilabel_tensor
        )

        curr_offset = 0

        for b in range(input_ids.size(0)):
            n_labels = num_labels_per_sample[b]
            if n_labels == 0:
                continue

            is_multi = is_multilabel_list[b]
            sample_logits = logits[curr_offset : curr_offset + n_labels]
            sample_targets = targets[curr_offset : curr_offset + n_labels]

            # Task-specific prediction logic
            if is_multi:
                # Multi-label: Independent thresholding
                sample_probs = torch.sigmoid(sample_logits)
                sample_preds = (sample_probs >= threshold).float()
            else:
                # Multi-class: Relative Argmax pick
                sample_probs = (
                    torch.softmax(sample_logits, dim=-1)
                    if sample_logits.numel() > 1
                    else torch.sigmoid(sample_logits)
                )
                sample_preds = torch.zeros_like(sample_logits)
                if sample_logits.numel() > 0:
                    pred_idx = torch.argmax(sample_logits)
                    sample_preds[pred_idx] = 1.0

            # Accumulate confusion matrix values per sample
            tp += (sample_preds * sample_targets).sum()
            fp += (sample_preds * (1.0 - sample_targets)).sum()
            fn += ((1.0 - sample_preds) * sample_targets).sum()
            tn += ((1.0 - sample_preds) * (1.0 - sample_targets)).sum()

            # Format visual sample logs for printing
            if (
                len(sample_predictions) < num_samples_to_print
                and accelerator.is_main_process
            ):
                ids = input_ids[b].tolist()
                full_text = tokenizer.decode(ids, skip_special_tokens=False)

                if special_tokens.SEP_TEXT in full_text:
                    text = (
                        full_text.split(special_tokens.SEP_TEXT)[-1]
                        .replace(tokenizer.pad_token, "")
                        .strip()
                    )
                else:
                    text = full_text.replace(tokenizer.pad_token, "").strip()

                labels = []
                for i, token_id in enumerate(ids):
                    if token_id == label_token_id:
                        label_tokens = []
                        for j in range(i + 1, len(ids)):
                            if ids[j] in special_ids_set:
                                break
                            label_tokens.append(ids[j])
                        decoded_label = tokenizer.decode(label_tokens).strip()
                        labels.append(decoded_label if decoded_label else "[UNK]")

                labels = labels[:n_labels]
                sample_targets_list = sample_targets.tolist()
                sample_probs_list = sample_probs.tolist()
                sample_preds_list = sample_preds.tolist()

                ground_truth_labels = [
                    lbl
                    for lbl, tgt in zip(labels, sample_targets_list)
                    if tgt == 1.0
                ]

                predicted_labels = []
                confidences = []

                for lbl, prob, pred in zip(
                    labels, sample_probs_list, sample_preds_list
                ):
                    if pred == 1.0:
                        predicted_labels.append(lbl)
                        confidences.append(round(prob, 4))

                sample_predictions.append(
                    {
                        "type": "Multi-Label" if is_multi else "Multi-Class",
                        "text": text,
                        "candidate_labels": labels,
                        "ground_truth_labels": ground_truth_labels,
                        "predicted_labels": predicted_labels,
                        "confidence": confidences,
                    }
                )

            curr_offset += n_labels

    # Gather metrics across GPU processes
    total_loss = accelerator.reduce(total_loss, reduction="sum").item()
    total_samples = accelerator.reduce(total_samples, reduction="sum").item()

    tp = accelerator.reduce(tp, reduction="sum").item()
    fp = accelerator.reduce(fp, reduction="sum").item()
    fn = accelerator.reduce(fn, reduction="sum").item()
    tn = accelerator.reduce(tn, reduction="sum").item()

    # Print representative predictions
    if accelerator.is_main_process:
        accelerator.print("\n" + "=" * 30 + " EVALUATION SAMPLES " + "=" * 30)
        for idx, sample in enumerate(sample_predictions, 1):
            accelerator.print(f"\nSample {idx} [{sample['type']}]:")
            accelerator.print(f"  Text             : {sample['text']}")
            accelerator.print(f"  Candidates       : {sample['candidate_labels']}")
            accelerator.print(f"  Ground Truth     : {sample['ground_truth_labels']}")
            accelerator.print(f"  Predicted        : {sample['predicted_labels']}")
            accelerator.print(f"  Confidence       : {sample['confidence']}")
        accelerator.print("=" * 80 + "\n")

    avg_loss = total_loss / max(total_samples, 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

    return {
        "val_loss": round(avg_loss, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4),
    }

def main():
    set_seed(42)

    use_fp16 = "fp16" if torch.cuda.is_available() else "no"
    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)

    # Pass ddp_kwargs handler into Accelerator
    accelerator = Accelerator(
        mixed_precision=use_fp16,
        dataloader_config=dataloader_config,
        kwargs_handlers=[ddp_kwargs],
    )
    special_tokens = SpecialTokens()

    model_name = "answerdotai/ModernBERT-base"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)

    num_added_tokens = tokenizer.add_special_tokens(
        {"additional_special_tokens": special_tokens.SPECIAL_TOKENS}
    )

    if num_added_tokens > 0:
        base_model.resize_token_embeddings(len(tokenizer))

    label_token_id = tokenizer.convert_tokens_to_ids(special_tokens.L_TOKEN)
    sep_text_token_id = tokenizer.convert_tokens_to_ids(special_tokens.SEP_TEXT)
    model = GLiNERTextClassifier(
        base_model=base_model, label_token_id=label_token_id,sep_text_token_id=sep_text_token_id
    )
    if accelerator.is_main_process:
        print(f"Added {num_added_tokens} new tokens to tokenizer.")

    train_data = ParquetIterableClassificationDataset(
        "/kaggle/working/multilabel_dataset_train.parquet",
        tokenizer=tokenizer,
        special_tokens=special_tokens,
    )

    train_loader = DataLoader(
        train_data,
        batch_size=48,
        collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
    )

    val_data = ParquetIterableClassificationDataset(
        "/kaggle/working/multilabel_dataset_val.parquet",
        tokenizer=tokenizer,
        special_tokens=special_tokens,
        max_negatives=15,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=64,
        collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
    )

    optimizer = torch.optim.AdamW(
    [
        {
            "params": model.encoder.parameters(),
            "lr": 2e-5,
            "weight_decay": 0.01,
        },
        {
            "params": model.text_proj.parameters(),
            "lr": 1e-5,
            "weight_decay": 0.01,
        },
        {
            "params": model.label_proj.parameters(),
            "lr": 1e-5,
            "weight_decay": 0.01,
        },
        {
            "params": [model.logit_scale],
            "lr": 1e-5,
            "weight_decay": 0.0,
        },
    ]
)
    criterion = HybridClassificationLoss()

    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )

    num_epochs = 2
    total_rows = len(train_data)
    per_process_rows = math.ceil(total_rows / accelerator.num_processes)
    total_train_batches = math.ceil(per_process_rows / 48)

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        progress_bar = tqdm(
            train_loader,
            total=total_train_batches,
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            disable=not accelerator.is_local_main_process,
        )

        for step, batch in enumerate(progress_bar):
            optimizer.zero_grad()

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            loss = criterion(
                logits=logits,
                targets=batch["targets"],
                num_labels_per_sample=batch["num_labels_per_sample"],
                is_multilabel=batch["is_multilabel"],
            )

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()
            current_avg_loss = total_train_loss / (step + 1)
            progress_bar.set_postfix({"loss": f"{current_avg_loss:.4f}"})

        val_metrics = evaluate(
            model=model,
            val_dataloader=val_loader,
            accelerator=accelerator,
            special_tokens=special_tokens,
            tokenizer=tokenizer,
            threshold=0.5,
            criterion=criterion,
            num_samples_to_print=20,
        )
        accelerator.print(f"Epoch {epoch + 1} Validation Metrics: {val_metrics}")

    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    if accelerator.is_main_process:
        torch.save(unwrapped_model.state_dict(), "gliner_classifier.pt")
        tokenizer.save_pretrained("./tokenizer")
        accelerator.print("Model successfully saved!")


if __name__ == "__main__":
    main()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

Added 10 new tokens to tokenizer.


Epoch 1/2:   0%|          | 0/2955 [00:00<?, ?it/s]


============================== EVALUATION SAMPLES ==============================

Sample 1 [Multi-Class]:
  Text             : Is there any other time period that has been so exhaustively covered by television (or the media in general) as the 1960s? No. And do we really need yet another trip through that turbulent time? Not really. But if we must have one, does it have to be as shallow as "The '60s"? <br /><br />I like to think that co-writers Bill Couturie and Robert Greenfield had more in mind for this two-part miniseries than what ultimately resulted, especially given Couturie's involvement in the superb HBO movie "Dear America: Letters Home From Vietnam" which utilized little original music and no original footage, letting the sights and sounds of the time speak for themselves. This presentation intercuts file footage with the dramatic production, but it doesn't do anyone any favours by trying to do too much in too little time; like so many of its ilk, it's seen from the point of 

Epoch 2/2:   0%|          | 0/2955 [00:00<?, ?it/s]


============================== EVALUATION SAMPLES ==============================

Sample 1 [Multi-Class]:
  Text             : Is there any other time period that has been so exhaustively covered by television (or the media in general) as the 1960s? No. And do we really need yet another trip through that turbulent time? Not really. But if we must have one, does it have to be as shallow as "The '60s"? <br /><br />I like to think that co-writers Bill Couturie and Robert Greenfield had more in mind for this two-part miniseries than what ultimately resulted, especially given Couturie's involvement in the superb HBO movie "Dear America: Letters Home From Vietnam" which utilized little original music and no original footage, letting the sights and sounds of the time speak for themselves. This presentation intercuts file footage with the dramatic production, but it doesn't do anyone any favours by trying to do too much in too little time; like so many of its ilk, it's seen from the point of 